# Pyspark intro & Ex 0 (a-j)
 
Pyspark is the python API to Apache Spark 

A short note on notebooks in databricks
- SQL notebook
- Python notebook 

The choice makes all cells defailt to that choice eg. SQL -> cells become SQL by default 


In [0]:

DATA_PATH = "/Volumes/data/olympic_games/raw_data/"

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True)
df_athletes


In [0]:
df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

In [0]:
(df_athletes.count(), len(df_athletes.columns))

## Infer the schema

We note that age, height becomes strings, that is because it has null values 

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True, inferSchema=True)
df_athletes


In [0]:
display(df_athletes)

### Define explicit schema

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema =spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True, schema=schema)
display(df_athletes_schema)

# EDA

## PySpark transformations

In [0]:
df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE', 'FIN', 'NOR', 'DEN', 'ISL') AND Medal != 'NA'"
).sort("NOC", "Medal").display()

In [0]:
nordic_countries_medals = spark.sql("""
    SELECT 
        NOC, 
        COUNT(medal) AS medal_count
    FROM 
        df_athletes_schema
    WHERE 
        Medal != 'NA' AND
        NOC IN ('SWE', 'FIN', 'NOR', 'DEN', 'ISL')
    GROUP BY 
        NOC
    ORDER BY
        medal_count DESC
""")

display(nordic_countries_medals)

Databricks visualization. Run in Databricks to view.

In [0]:
all_countries_medals = spark.sql("""
    SELECT 
        NOC, 
        COUNT(medal) AS medal_count
    FROM 
        df_athletes_schema
    WHERE 
        Medal != 'NA' AND len(NOC) = 3
    GROUP BY 
        NOC
    ORDER BY
        medal_count DESC
""")

display(all_countries_medals)

Databricks visualization. Run in Databricks to view.

### Spark SQL 

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

spark.sql(
    "SELECT * FROM df_athletes_schema WHERE NOC = 'SWE' AND Sport = 'Table Tennis' AND Medal != 'NA'"
).display()

# To do pick out swedish medals in table tennis

## Count medals and plot

In [0]:
df_swe_medals = spark.sql("""
          SELECT 
          sport, COUNT(medal) AS medal_count
          FROM df_athletes_schema
          WHERE NOC = 'SWE' AND medal IN ('Bronze', 'Silver', 'Gold')
          GROUP BY sport 
          ORDER BY medal_count DESC
          """)

df_swe_medals.display()

In [0]:
fig = df_swe_medals.plot(kind="bar", y="sport", x="medal_count")
fig.update_layout(
    autosize=False,
    width=1000,
    height=500,
    title="Swedish Athletes in Table Tennis",
    xaxis_title="Medal Count",
    yaxis_title="Sport",
)
fig.show()

### Ingesting data to unity catalog


In [0]:
%sql
SHOW SCHEMAS IN data;

CREATE TABLE IF NOT EXISTS 
data.olympic_games.sweden_medals AS 
(
    SELECT 
        name, 
        age, 
        height, 
        weight, 
        year, 
        sport,
        medal 
    FROM df_athletes_schema
    WHERE 
        NOC = 'SWE' AND 
        medal in ('Gold', 'Silver','Bronze')
)

In [0]:
%sql
SELECT name, sport, age
FROM data.olympic_games.sweden_medals
WHERE sport = 'Swimming' and age > 24


##### B - Use spark columns method to find out the columns

In [0]:
#Returns list with column names 
df_athletes.columns

In [0]:
# Shows in databricks 
display(df_athletes.columns)

##### C -  Find out the 10 oldest atheletes, their age and the sport

In [0]:
oldest_athletes_df = spark.sql("""
    SELECT DISTINCT
        name, 
        sport, 
        age
    FROM 
        df_athletes_schema
    SORT BY 
        age DESC
    LIMIT 10; 
""")

display(oldest_athletes_df)

In [0]:
youngest_athletes_df = spark.sql("""
    SELECT DISTINCT
        name, 
        sport, 
        age
    FROM 
        df_athletes_schema
    WHERE 
        age IS NOT NULL
    SORT BY 
        age ASC
    LIMIT 10; 
""")

display(youngest_athletes_df)

##### E - Find out the five sports with highest median age

In [0]:
highest_median_age_df = spark.sql("""
    SELECT
        sport, 
        median(age) as median_age
    FROM 
        df_athletes_schema
    WHERE 
        age IS NOT NULL
    GROUP BY 
        sport
    ORDER BY 
        median_age DESC
    LIMIT 5
""")

display(highest_median_age_df)

#####   F- Find out the five sports with lowest median age

In [0]:
lowest_median_age_df = spark.sql("""
    SELECT 
        sport, 
        median(age) as median_age 
    FROM 
        df_athletes_schema
    WHERE
        age IS NOT NULL
    GROUP BY 
        sport 
    ORDER BY 
        median_age ASC
    LIMIT 5
    
""")

display(lowest_median_age_df)


In [0]:
top_10_gold_countries_df = spark.sql("""
    SELECT
        NOC AS Country, 
        COUNT(medal) as gold_medals
    FROM 
        df_athletes_schema
    WHERE 
        medal = 'Gold' AND medal IS NOT NULL
    GROUP BY 
        NOC
    ORDER BY 
        gold_medals DESC
    LIMIT 10
""")
display(top_10_gold_countries_df)

##### H - Find out top 10 countries after number of medals

In [0]:
top_10_medals_countries_df = spark.sql("""
    SELECT
        NOC AS Country,
        COUNT(medal) AS Total_Medals 
    FROM 
        df_athletes_schema
    WHERE 
        medal IN ('Gold', 'Silver', 'Bronze') AND medal IS NOT NULL
    GROUP BY 
        NOC
    ORDER BY 
        total_medals DESC
    LIMIT 10
""")
display(top_10_medals_countries_df)

##### I - Plot a time series line chart of number of female and male atheletes in same graph.

In [0]:

male_female_df = spark.sql("""
    SELECT 
        year,
        SUM(CASE WHEN sex = 'M' THEN 1 ELSE 0 END) as male_count,
        SUM(CASE WHEN sex = 'F' THEN 1 ELSE 0 END) as female_count,
        COUNT(sex) AS total_count
    FROM 
        df_athletes_schema
    WHERE
        year IS NOT NULL
    GROUP BY 
        year
    ORDER BY 
        year
""")

display(male_female_df)





Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
fig = male_female_df.plot(
    kind="line",
    y=["female_count", "male_count"],
    x="year"
)

fig.update_layout(
    autosize=True,
    title="Male and Female Participation Over Time",
    xaxis_title="Year",
    yaxis_title="Number of Participants",
    legend_title="Sex",
)

fig.show()

In [0]:
fig = male_female_df.plot(
    kind="line",
    y=["female_count", "male_count", "total_count"],
    x="year"
)

fig.update_layout(
    autosize=True,
    title="Male and Female Participation Over Time",
    xaxis_title="Year",
    yaxis_title="Number of Participants",
    legend_title="Sex",
)

fig.show()